# Intervention Plans Pipeline (Ch. 1-17)

This notebook implements all required analytics phases:
1. Problem framing
2. Data acquisition and preparation
3. Exploration
4. Modeling (explanatory + predictive)
5. Evaluation and selection
6. Feature selection
7. Deployment artifact export

The notebook is reproducible and writes model artifacts to `ml-pipelines/artifacts/`.

## Split Version (Phase-by-Phase Cells)
Run the cells below in order for a step-by-step pipeline.

In [5]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PIPELINE_NAME = "intervention_plans"
DATA_PATH = Path("datasets/intervention_plans.csv")
if not DATA_PATH.exists():
    alt_data_path = Path("..") / DATA_PATH
    if alt_data_path.exists():
        DATA_PATH = alt_data_path
ARTIFACT_DIR = Path("ml-pipelines/artifacts")
if not ARTIFACT_DIR.parent.exists():
    ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

In [6]:
# 1) Problem Framing

def choose_target(df: pd.DataFrame) -> str:
    candidates = [
        c for c in df.columns
        if df[c].notna().sum() >= int(0.7 * len(df)) and df[c].nunique(dropna=True) > 5
    ]
    priority_terms = ["status", "score", "risk", "outcome", "priority", "duration", "cost", "amount", "total", "count"]
    ranked = sorted(
        candidates,
        key=lambda c: (
            not any(term in c.lower() for term in priority_terms),
            df[c].dtype == "O",
            -df[c].nunique(dropna=True),
        ),
    )
    if not ranked:
        raise ValueError("No suitable target variable found with enough variation.")
    return ranked[0]


def classify_problem(y: pd.Series) -> str:
    nunique = y.nunique(dropna=True)
    if y.dtype == "O" or str(y.dtype).startswith("category"):
        return "classification"
    if nunique <= 10:
        return "classification"
    return "regression"

df_raw = pd.read_csv(DATA_PATH)
target_col = choose_target(df_raw)
problem_type = classify_problem(df_raw[target_col])

print(f"Loaded {PIPELINE_NAME}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
display(df_raw.head(3))
print(f"Business question: Which factors influence '{target_col}', and how accurately can we predict it?")
print(f"Modeling approach: explanatory + predictive ({problem_type})")

Loaded intervention_plans: 180 rows x 11 cols


,plan_id,resident_id,plan_category,plan_description,services_provided,target_value,target_date,status,case_conference_date,created_at,updated_at
0,1,1,Safety,Maintain a stable and safe environment,"Healing, Legal Services, Teaching",4.20,2024-02-01,On Hold,2023-11-01,2023-10-01 00:00:00,2024-03-01 00:00:00
1,2,1,Education,Improve participation and course completion,"Caring, Legal Services, Healing",0.85,2024-02-01,In Progress,2024-01-30,2023-10-01 00:00:00,2024-03-01 00:00:00
2,3,1,Physical Health,Improve nutrition and overall wellbeing,"Teaching, Healing, Caring",4.20,2024-02-01,On Hold,2023-10-24,2023-10-01 00:00:00,2024-03-01 00:00:00


Business question: Which factors influence 'plan_id', and how accurately can we predict it?
Modeling approach: explanatory + predictive (regression)


In [7]:
# 2) Data Acquisition and Preparation

df = df_raw.copy()
y = df[target_col]
X = df.drop(columns=[target_col])

id_like_cols = [
    c for c in X.columns
    if any(tok in c.lower() for tok in ["id", "uuid", "record"]) and X[c].nunique(dropna=True) > 0.8 * len(X)
]
if id_like_cols:
    X = X.drop(columns=id_like_cols)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)],
    remainder="drop",
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,
    stratify=y if problem_type == "classification" and y.nunique(dropna=True) <= 20 else None,
)

print(f"Prepared train/test split: {X_train.shape} / {X_test.shape}")

Prepared train/test split: (144, 10) / (36, 10)


In [8]:
# 3) Exploration
print("Missingness (%):")
display((df.isna().mean() * 100).sort_values(ascending=False).head(10).to_frame("missing_pct"))
print("Target distribution:")
if problem_type == "classification":
    display(y.value_counts(dropna=False).to_frame("count"))
else:
    display(y.describe().to_frame("target_summary"))

Missingness (%):


,missing_pct
case_conference_date,26.666667
plan_id,0.000000
resident_id,0.000000
plan_category,0.000000
plan_description,0.000000
services_provided,0.000000
target_value,0.000000
target_date,0.000000
status,0.000000
created_at,0.000000


Target distribution:


,target_summary
count,180.000000
mean,90.500000
std,52.105662
min,1.000000
25%,45.750000
50%,90.500000
75%,135.250000
max,180.000000


In [9]:
# 4) Modeling (Explanatory + Predictive)
if problem_type == "classification":
    explanatory_estimator = LogisticRegression(max_iter=2000)
    predictive_base = RandomForestClassifier(random_state=RANDOM_STATE)
    predictive_alt = GradientBoostingClassifier(random_state=RANDOM_STATE)
    score_metric = "f1_weighted"
else:
    explanatory_estimator = LinearRegression()
    predictive_base = RandomForestRegressor(random_state=RANDOM_STATE)
    predictive_alt = GradientBoostingRegressor(random_state=RANDOM_STATE)
    score_metric = "neg_root_mean_squared_error"

grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_leaf": [1, 3, 8],
}

explanatory_model = Pipeline(steps=[("prep", preprocessor), ("model", explanatory_estimator)])
explanatory_model.fit(X_train, y_train)

predictive_pipe = Pipeline(steps=[("prep", preprocessor), ("model", predictive_base)])
grid_search = GridSearchCV(predictive_pipe, grid, scoring=score_metric, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)
predictive_model = grid_search.best_estimator_

alt_model = Pipeline(steps=[("prep", preprocessor), ("model", predictive_alt)])
alt_model.fit(X_train, y_train)

print("Models trained.")

Models trained.


In [10]:
# 5) Evaluation and Selection
if problem_type == "classification":
    y_pred_exp = explanatory_model.predict(X_test)
    y_pred_pred = predictive_model.predict(X_test)
    y_pred_alt = alt_model.predict(X_test)
    eval_rows = [
        {"model": "explanatory_logistic", "accuracy": accuracy_score(y_test, y_pred_exp), "f1_weighted": f1_score(y_test, y_pred_exp, average="weighted")},
        {"model": "predictive_random_forest_tuned", "accuracy": accuracy_score(y_test, y_pred_pred), "f1_weighted": f1_score(y_test, y_pred_pred, average="weighted")},
        {"model": "predictive_gradient_boosting", "accuracy": accuracy_score(y_test, y_pred_alt), "f1_weighted": f1_score(y_test, y_pred_alt, average="weighted")},
    ]
    for row, model_obj in [(eval_rows[0], explanatory_model), (eval_rows[1], predictive_model), (eval_rows[2], alt_model)]:
        if hasattr(model_obj, "predict_proba"):
            proba = model_obj.predict_proba(X_test)
            try:
                row["roc_auc"] = roc_auc_score(y_test, proba[:, 1]) if proba.shape[1] == 2 else roc_auc_score(y_test, proba, multi_class="ovr")
            except Exception:
                row["roc_auc"] = np.nan
else:
    y_pred_exp = explanatory_model.predict(X_test)
    y_pred_pred = predictive_model.predict(X_test)
    y_pred_alt = alt_model.predict(X_test)
    rmse = lambda yt, yp: float(np.sqrt(mean_squared_error(yt, yp)))
    eval_rows = [
        {"model": "explanatory_linear", "rmse": rmse(y_test, y_pred_exp), "mae": mean_absolute_error(y_test, y_pred_exp), "r2": r2_score(y_test, y_pred_exp)},
        {"model": "predictive_random_forest_tuned", "rmse": rmse(y_test, y_pred_pred), "mae": mean_absolute_error(y_test, y_pred_pred), "r2": r2_score(y_test, y_pred_pred)},
        {"model": "predictive_gradient_boosting", "rmse": rmse(y_test, y_pred_alt), "mae": mean_absolute_error(y_test, y_pred_alt), "r2": r2_score(y_test, y_pred_alt)},
    ]

eval_df = pd.DataFrame(eval_rows)
display(eval_df)

fairness_rows = []
group_candidates = [c for c in cat_cols if 2 <= X_test[c].nunique(dropna=True) <= 10]
if group_candidates:
    group_col = group_candidates[0]
    test_with_preds = X_test.copy()
    test_with_preds["__y_true__"] = y_test.values
    test_with_preds["__y_pred__"] = y_pred_pred
    for grp, gdf in test_with_preds.groupby(group_col):
        if len(gdf) < 5:
            continue
        fairness_rows.append(
            {"group_col": group_col, "group": grp, "size": len(gdf), "accuracy": accuracy_score(gdf["__y_true__"], gdf["__y_pred__"])}
            if problem_type == "classification"
            else {"group_col": group_col, "group": grp, "size": len(gdf), "mae": mean_absolute_error(gdf["__y_true__"], gdf["__y_pred__"])}
        )
fairness_df = pd.DataFrame(fairness_rows)
if not fairness_df.empty:
    display(fairness_df)

,model,rmse,mae,r2
0,explanatory_linear,0.784440,0.578526,0.999759
1,predictive_random_forest_tuned,1.545229,1.318750,0.999065
2,predictive_gradient_boosting,0.607548,0.528444,0.999855


,group_col,group,size,mae
0,plan_category,Education,10,0.825083
1,plan_category,Physical Health,8,1.927292
2,plan_category,Safety,18,1.322546


In [11]:
# 6) Feature Selection + Recommendations
feature_names = predictive_model.named_steps["prep"].get_feature_names_out()
importances = predictive_model.named_steps["model"].feature_importances_
fi_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False).head(15).reset_index(drop=True)
display(fi_df)

coef_df = pd.DataFrame()
if hasattr(explanatory_model.named_steps["model"], "coef_"):
    coef = explanatory_model.named_steps["model"].coef_
    coef = np.mean(np.abs(coef), axis=0) if np.ndim(coef) > 1 else np.abs(coef)
    coef_df = pd.DataFrame({"feature": feature_names, "coef_abs": coef}).sort_values("coef_abs", ascending=False).head(15).reset_index(drop=True)
    display(coef_df)

recommendations = [
    f"Prioritize interventions on records with high values of: {', '.join(fi_df['feature'].head(3).tolist())}.",
    "Use explanatory coefficients for policy discussion and stakeholder communication.",
    "Track fairness gaps every retraining cycle and adjust data collection accordingly.",
]
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

,feature,importance
0,num__resident_id,0.998347
1,cat__status_In Progress,0.000088
2,cat__case_conference_date_2023-09-25,0.000058
3,cat__updated_at_2023-12-01 00:00:00,0.000049
4,cat__status_Open,0.000038
5,cat__created_at_2024-01-01 00:00:00,0.000037
6,cat__created_at_2023-10-01 00:00:00,0.000034
7,cat__target_date_2024-08-01,0.000030
8,cat__target_date_2023-11-01,0.000029
9,cat__status_Achieved,0.000028


,feature,coef_abs
0,num__resident_id,51.584778
1,cat__created_at_2023-08-01 00:00:00,1.141173
2,cat__case_conference_date_2023-12-15,1.141173
3,cat__case_conference_date_2024-07-06,1.118282
4,"cat__services_provided_Healing, Caring, Teaching",1.099753
5,cat__case_conference_date_2024-05-19,1.095887
6,cat__case_conference_date_2023-10-24,1.090524
7,cat__created_at_2023-01-01 00:00:00,1.012281
8,cat__case_conference_date_2024-08-21,0.900959
9,cat__created_at_2025-02-01 00:00:00,0.897686


1. Prioritize interventions on records with high values of: num__resident_id, cat__status_In Progress, cat__case_conference_date_2023-09-25.
2. Use explanatory coefficients for policy discussion and stakeholder communication.
3. Track fairness gaps every retraining cycle and adjust data collection accordingly.


In [12]:
# 7) Deployment Artifacts
predictive_artifact_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_predictive_model.joblib"
explanatory_artifact_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_explanatory_model.joblib"
metrics_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_model_metrics.csv"
fairness_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_fairness_report.csv"
fi_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_top_features.csv"
coef_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_top_explanatory_coefficients.csv"
schema_path = ARTIFACT_DIR / f"{PIPELINE_NAME}_model_schema.json"

joblib.dump(predictive_model, predictive_artifact_path)
joblib.dump(explanatory_model, explanatory_artifact_path)
eval_df.to_csv(metrics_path, index=False)
if not fairness_df.empty:
    fairness_df.to_csv(fairness_path, index=False)
fi_df.to_csv(fi_path, index=False)
if not coef_df.empty:
    coef_df.to_csv(coef_path, index=False)

schema = {
    "pipeline": PIPELINE_NAME,
    "dataset": str(DATA_PATH),
    "target": target_col,
    "problem_type": problem_type,
    "features_used": X.columns.tolist(),
    "id_like_columns_dropped": id_like_cols,
    "best_predictive_model": str(predictive_model.named_steps["model"]),
    "evaluation_rows": eval_rows,
    "recommendations": recommendations,
}
schema_path.write_text(json.dumps(schema, indent=2), encoding="utf-8")

print("Saved:")
for p in [predictive_artifact_path, explanatory_artifact_path, metrics_path, fi_path, schema_path]:
    print(f"- {p}")
if not fairness_df.empty:
    print(f"- {fairness_path}")
if not coef_df.empty:
    print(f"- {coef_path}")

Saved:
- ml-pipelines\artifacts\intervention_plans_predictive_model.joblib
- ml-pipelines\artifacts\intervention_plans_explanatory_model.joblib
- ml-pipelines\artifacts\intervention_plans_model_metrics.csv
- ml-pipelines\artifacts\intervention_plans_top_features.csv
- ml-pipelines\artifacts\intervention_plans_model_schema.json
- ml-pipelines\artifacts\intervention_plans_fairness_report.csv
- ml-pipelines\artifacts\intervention_plans_top_explanatory_coefficients.csv
